# Aktivitas Naive Bayes — Apakah Cocok Bermain Tenis Hari Ini?

**Mata Kuliah:** DSB07 — Machine Learning  
**Topik:** Naive Bayes (Pertemuan 10)  
**Durasi:** 45 menit  

## Anggota Kelompok
_Isi nama & NIM di bawah ini sebelum mulai:_
1. ...
2. ...
3. ...
4. ...

## Petunjuk
Lengkapi setiap sel berlabel `# TODO`. Jangan ubah sel yang sudah berisi kode lengkap kecuali diminta.
Jalankan sel secara berurutan dari atas ke bawah.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

pd.set_option("display.precision", 4)
np.set_printoptions(precision=4, suppress=True)

## Tahap 1 — Memahami Data (5 menit)

Dataset `play_tennis` berisi 14 hari pengamatan apakah cocok untuk bermain tenis berdasarkan kondisi cuaca.

| Fitur | Nilai mungkin |
|-------|---------------|
| Outlook | Sunny, Overcast, Rain |
| Temperature | Hot, Mild, Cool |
| Humidity | High, Normal |
| Wind | Weak, Strong |
| **Play** (label) | **Yes, No** |

In [ ]:
# Ganti USERNAME dengan akun GitHub dosen Anda setelah repo di-push.
url = "https://raw.githubusercontent.com/USERNAME/naive-bayes/main/data/play_tennis.csv"
df = pd.read_csv(url)
df

In [ ]:
# TODO 1: Tampilkan distribusi label 'Play' (berapa Yes, berapa No)
# Hint: gunakan .value_counts() pada kolom 'Play'
...

## Tahap 2 — Hitung Manual (15 menit)

**Kasus uji:** Hari ini cuaca **Sunny**, suhu **Cool**, kelembapan **High**, angin **Strong**.  
**Pertanyaan:** Apakah cocok bermain tenis (`Play = Yes` atau `No`)?

Rumus (slide 2.2):
$$P(K \mid F_1, F_2, \dots, F_n) \propto P(K) \times \prod_{i=1}^{n} P(F_i \mid K)$$

Kita akan menghitung *tanpa* Laplace smoothing terlebih dahulu (sesuai rumus di slide).

In [ ]:
test = {"Outlook": "Sunny", "Temperature": "Cool", "Humidity": "High", "Wind": "Strong"}
test

### 2.1 Probabilitas Prior P(K)

In [ ]:
# TODO 2: Hitung P(Yes) dan P(No)
n_total = ...   # total baris
n_yes = ...     # jumlah Play == 'Yes'
n_no = ...      # jumlah Play == 'No'

p_yes = ...
p_no = ...

print(f"P(Yes) = {p_yes:.4f}")
print(f"P(No)  = {p_no:.4f}")

### 2.2 Probabilitas Kondisional P(F_i | K)

Untuk setiap fitur dalam kasus uji, hitung berapa kali nilainya muncul dalam tiap kelas, dibagi total baris kelas tersebut.

In [ ]:
df_yes = df[df["Play"] == "Yes"]
df_no = df[df["Play"] == "No"]

# TODO 3: Hitung 4 probabilitas kondisional untuk kelas Yes
# Contoh: p_sunny_yes = (df_yes['Outlook'] == 'Sunny').sum() / len(df_yes)
p_sunny_yes  = ...
p_cool_yes   = ...
p_high_yes   = ...
p_strong_yes = ...

# TODO 4: Hitung 4 probabilitas kondisional untuk kelas No
p_sunny_no  = ...
p_cool_no   = ...
p_high_no   = ...
p_strong_no = ...

print("Kelas Yes:", p_sunny_yes, p_cool_yes, p_high_yes, p_strong_yes)
print("Kelas No :", p_sunny_no,  p_cool_no,  p_high_no,  p_strong_no)

### 2.3 Posterior & Keputusan

In [ ]:
# TODO 5: Hitung skor posterior (proporsional, belum dinormalisasi)
score_yes = ...   # p_yes * p_sunny_yes * p_cool_yes * p_high_yes * p_strong_yes
score_no  = ...   # p_no  * p_sunny_no  * p_cool_no  * p_high_no  * p_strong_no

# TODO 6: Normalisasi sehingga total = 1
total = ...
post_yes = ...
post_no  = ...

prediksi_manual = "Yes" if post_yes > post_no else "No"

print(f"P(Yes | x) = {post_yes:.4f}")
print(f"P(No  | x) = {post_no:.4f}")
print(f"Prediksi manual: {prediksi_manual}")

## Tahap 3 — Implementasi dengan scikit-learn (15 menit)

`CategoricalNB` di sklearn membutuhkan input numerik, jadi fitur kategorikal kita ubah dulu dengan `OrdinalEncoder`.

In [ ]:
features = ["Outlook", "Temperature", "Humidity", "Wind"]

encoder = OrdinalEncoder()
X = encoder.fit_transform(df[features])
y = df["Play"].values

print("Pemetaan kategori -> angka:")
for col, cats in zip(features, encoder.categories_):
    print(f"  {col}: {dict(enumerate(cats))}")

In [ ]:
# TODO 7: Latih CategoricalNB (alpha default = 1, pakai Laplace smoothing)
model = CategoricalNB()
# model.fit(...)

# TODO 8: Hitung akurasi pada data latih (perlu .predict(X) dulu)
y_pred = ...
akurasi = ...
print(f"Akurasi pada data latih: {akurasi:.4f}")

In [ ]:
# TODO 9: Prediksi kasus uji yang SAMA dengan Tahap 2
x_test_df = pd.DataFrame([test])
x_test = encoder.transform(x_test_df[features])

prediksi_sklearn = ...   # gunakan model.predict
proba_sklearn = ...      # gunakan model.predict_proba

print(f"Prediksi sklearn : {prediksi_sklearn[0]}")
print(f"Kelas-kelas      : {model.classes_}")
print(f"Probabilitas     : {proba_sklearn[0]}")

## Tahap 4 — Diskusi Kelompok (7 menit)

_Tulis jawaban kelompok dalam sel di bawah._

1. Apakah **prediksi kelas** manual sama dengan sklearn? Apakah **nilai probabilitas**-nya juga sama? Jika berbeda, kira-kira kenapa?  
   _Hint: cari arti parameter `alpha` di `CategoricalNB` (Laplace / additive smoothing)._
2. Coba ubah `CategoricalNB(alpha=1e-10)` dan jalankan ulang Tahap 3. Apakah probabilitas sklearn jadi lebih dekat ke hasil manual?
3. Apa yang akan terjadi jika kasus uji mengandung kombinasi (kategori, kelas) yang **tidak pernah muncul** di data latih (misalnya `Outlook=Overcast` pada kelas `No`)? Bagaimana Laplace smoothing menyelamatkan situasi ini?  
   _Hint: lihat slide 1.3 poin #3 (Sensitif terhadap Atribut yang Hilang)._
4. Naive Bayes mengasumsikan **semua fitur saling independen**. Menurut kelompok Anda, apakah `Humidity` dan `Outlook` benar-benar independen di dunia nyata? Apa konsekuensinya untuk akurasi model?

**Jawaban Kelompok:**

1. ...
2. ...
3. ...
4. ...

## Tahap 5 — Refleksi (3 menit)

Tulis 3 kalimat singkat:
- Hal terpenting yang saya pelajari hari ini adalah ...
- Bagian paling sulit adalah ...
- Naive Bayes cocok dipakai ketika ...

## Submission
Simpan notebook ini ke Google Drive masing-masing, lalu kumpulkan link **share** (akses *Anyone with the link → Viewer*) ke kanal kelas yang ditentukan dosen.